# Logit and Probit Models: Probabilistic Modeling for Binary Outcomes

## 1. Introduction and Overview

The fundamental challenge in binary classification is mapping an unconstrained linear combination of inputs Z, which ranges from negative infinity to positive infinity, into a strictly bounded probability space P between 0 and 1. Logit and Probit models achieve this by using specific Cumulative Distribution Functions (CDFs) as link functions to squash linear predictions into valid probabilities.

When modeling binary outcomes (e.g., predicting if a user will click an ad, if a patient has a disease, or if a transaction is fraudulent), we are attempting to estimate the probability that a target variable Y equals 1, given a feature vector X.

In this notebook, we will explore the theoretical foundations and computational implementations of the Linear Probability Model (LPM), Logit, and Probit models using Python.

In [ ]:
# Setup and Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import log_loss, accuracy_score, roc_auc_score
from scipy.optimize import minimize
import warnings

# Configure visualization and notebook settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set global random seed for complete reproducibility
np.random.seed(42)
print('Libraries imported successfully and random seed set to 42.')

## 2. Data Creation: Generating a Synthetic Binary Dataset

To thoroughly investigate these models, we need a dataset where we know the true underlying data-generating process. We will simulate a scenario predicting whether a beam breaks (Y=1) or holds (Y=0) based on applied Weight and age of the beam.

In [ ]:
# Simulate features: Weight (kg) and Age (years)
n_samples = 1500
weight = np.random.uniform(50, 200, n_samples)
age = np.random.uniform(1, 20, n_samples)

# True parameters for the Data Generating Process
true_intercept = -6.0
true_weight_coef = 0.04
true_age_coef = 0.15

# Calculate the unobservable Latent Variable Z (Linear Predictor)
Z = true_intercept + (weight * true_weight_coef) + (age * true_age_coef)

# Apply the Logistic CDF (Sigmoid) to get true probabilities
p_true = 1 / (1 + np.exp(-Z))

# Generate binary outcomes using Bernoulli trials based on p_true
beam_breaks = np.random.binomial(n=1, p=p_true)

# Create a DataFrame
df = pd.DataFrame({
    'Weight': weight,
    'Age': age,
    'Z_Latent': Z,
    'True_Prob': p_true,
    'Breaks': beam_breaks
})

print('Dataset Snapshot:')
print(df.head())
print(f'\nOverall Break Rate: {df["Breaks"].mean() * 100:.1f}%')

## 3. Core Concept 1: The Linear Probability Model (LPM)

The naive approach to binary classification is the Linear Probability Model (LPM):
P(Y=1 | X) = beta_0 + beta_1 * X_1 + ... + beta_k * X_k = X * beta

Let's fit an Ordinary Least Squares (OLS) model to this binary data and visualize why this is structurally flawed.

In [ ]:
# Fit an OLS (Linear Regression) model to the binary outcome
X_features = df[['Weight', 'Age']]
y_target = df['Breaks']
lpm_model = LinearRegression()
lpm_model.fit(X_features, y_target)

# Predict probabilities using the linear model
lpm_predictions = lpm_model.predict(X_features)

# Check for invalid probabilities
invalid_high = np.sum(lpm_predictions > 1)
invalid_low = np.sum(lpm_predictions < 0)

print(f'LPM predicted {invalid_high} probabilities > 1')
print(f'LPM predicted {invalid_low} probabilities < 0')

# Visualize the flaw using just the Weight feature (holding Age constant at mean)
mean_age = df['Age'].mean()
weight_range = np.linspace(0, 300, 300)
X_plot = pd.DataFrame({'Weight': weight_range, 'Age': np.full_like(weight_range, mean_age)})
lpm_plot_preds = lpm_model.predict(X_plot)

plt.figure(figsize=(10, 5))
plt.scatter(df['Weight'], df['Breaks'], alpha=0.1, color='gray', label='Observed Data')
plt.plot(weight_range, lpm_plot_preds, color='red', lw=3, label='LPM Prediction Line')
plt.axhline(1, color='black', linestyle='--', alpha=0.5)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.fill_between(weight_range, lpm_plot_preds, 1, where=(lpm_plot_preds > 1), color='red', alpha=0.2)
plt.fill_between(weight_range, lpm_plot_preds, 0, where=(lpm_plot_preds < 0), color='red', alpha=0.2)
plt.title('The Flaw of the Linear Probability Model (LPM)')
plt.xlabel('Weight (kg)')
plt.ylabel('Predicted Probability of Breaking')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Core Concept 2: The Logit and Probit Link Functions

To fix the unbounded nature of the LPM, we pass the linear combination Z = X * beta through a non-linear S-shaped curve (a CDF).

1. The Logit Model uses the Standard Logistic CDF: p = 1 / (1 + exp(-Z))
2. The Probit Model uses the Standard Normal CDF: p = integral from -inf to Z of N(0,1)

Let's visualize and compare these mathematical functions.

In [ ]:
# 1. Define the linear combination range (Z)
z_vals = np.linspace(-5, 5, 1000)

# 2. Logit Transformation (Sigmoid)
def logit_prob(z):
    return 1 / (1 + np.exp(-z))
p_logit = logit_prob(z_vals)

# 3. Probit Transformation (Standard Normal CDF)
def probit_prob(z):
    return norm.cdf(z)
p_probit = probit_prob(z_vals)

# 4. Visualization
plt.figure(figsize=(10, 6))
plt.plot(z_vals, p_logit, label='Logit (Logistic CDF)', color='#1f77b4', linewidth=3)
plt.plot(z_vals, p_probit, label='Probit (Normal CDF)', color='#ff7f0e', linewidth=3, linestyle='--')
plt.axvline(0, color='gray', linestyle=':', alpha=0.7)
plt.axhline(0.5, color='gray', linestyle=':', alpha=0.7)
plt.title('Logit vs Probit Link Functions')
plt.xlabel('Linear Predictor (Z = X * beta)')
plt.ylabel('Probability P(Y=1)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Observation: Probit approaches the 0 and 1 asymptotes faster than Logit. Logit has fatter tails.')

## 5. Implementing Logistic Regression with Scikit-Learn

In machine learning pipelines, we almost exclusively use the Logit model (LogisticRegression). Let's fit it to our synthetic dataset and extract the coefficients.

Note: Scikit-learn applies L2 regularization by default. To compare true statistical coefficients, we set penalty=None.

In [ ]:
# Fit unregularized Logistic Regression
logit_ml = LogisticRegression(penalty=None)
logit_ml.fit(X_features, y_target)

print('--- Scikit-Learn Logistic Regression Results ---')
print(f'True Intercept: {true_intercept} | Estimated: {logit_ml.intercept_[0]:.3f}')
print(f'True Weight Coef: {true_weight_coef} | Estimated: {logit_ml.coef_[0][0]:.3f}')
print(f'True Age Coef: {true_age_coef} | Estimated: {logit_ml.coef_[0][1]:.3f}')
print('\nThe unregularized model perfectly recovered the true data-generating parameters!')

## 6. Econometric Inference: Statsmodels (Logit vs Probit)

When we care about inference (p-values, standard errors), we use statsmodels. Let's fit both Logit and Probit models to the same data and compare their coefficients.

In [ ]:
# Statsmodels requires an explicit constant (intercept) column
X_sm = sm.add_constant(X_features)

# Fit Logit Model
sm_logit = sm.Logit(y_target, X_sm).fit(disp=False)

# Fit Probit Model
sm_probit = sm.Probit(y_target, X_sm).fit(disp=False)

# Combine coefficients for comparison
# Amemiya Scaling: Probit coefs * (pi / sqrt(3)) roughly equals Logit coefs
amemiya_factor = np.pi / np.sqrt(3)

coef_comparison = pd.DataFrame({
    'Logit_Coef': sm_logit.params,
    'Probit_Coef': sm_probit.params,
    'Scaled_Probit': sm_probit.params * amemiya_factor
})

print('--- Model Coefficient Comparison ---')
print(coef_comparison.round(4))
print('\nNotice how scaling the Probit coefficients brings them extremely close to the Logit coefficients.')

## 7. Core Concept 3: Odds Ratio Interpretation (Logit)

One of the most powerful aspects of the Logit model is its interpretation via odds. The log-odds in a logistic regression model are strictly linear with respect to the parameters.

ln(p / (1-p)) = beta_0 + beta_1 * X_1

Exponentiating the coefficient (e^beta_1) gives us the Odds Ratio. A 1-unit increase in X_1 multiplies the odds of the event happening by e^beta_1.

In [ ]:
print('--- Odds Ratio Interpretation ---')
for feature in ['Weight', 'Age']:
    beta = sm_logit.params[feature]
    odds_ratio = np.exp(beta)
    print(f'Feature: {feature}')
    print(f'  Log-Odds Coef (Beta): {beta:.4f}')
    print(f'  Odds Ratio (exp(Beta)): {odds_ratio:.4f}')
    
    percentage_change = (odds_ratio - 1) * 100
    print(f'  Interpretation: A 1-unit increase in {feature} increases the odds of breaking by {percentage_change:.2f}%\n')

## 8. Visualizing the Log-Odds vs Probability Space

Let's visually prove that while the relationship between a feature and Log-Odds is perfectly linear, its relationship with Probability is non-linear (the S-curve).

In [ ]:
# Generate a range for Weight, keeping Age constant
w_range = np.linspace(50, 250, 200)
z_linear = sm_logit.params['const'] + sm_logit.params['Weight']*w_range + sm_logit.params['Age']*mean_age
prob_nonlinear = logit_prob(z_linear)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear Log-Odds Space
axes[0].plot(w_range, z_linear, color='blue', lw=3)
axes[0].set_title('Log-Odds Space (Strictly Linear)')
axes[0].set_xlabel('Weight (kg)')
axes[0].set_ylabel('Log-Odds (Z)')
axes[0].grid(alpha=0.3)

# Non-Linear Probability Space
axes[1].plot(w_range, prob_nonlinear, color='green', lw=3)
axes[1].set_title('Probability Space (Non-Linear)')
axes[1].set_xlabel('Weight (kg)')
axes[1].set_ylabel('Probability P(Y=1)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Mathematical Estimation: Maximum Likelihood (MLE)

Unlike OLS which minimizes Mean Squared Error, Logit and Probit models are fit using Maximum Likelihood Estimation (MLE).
The Log-Likelihood is mathematically equivalent to the negative Cross-Entropy loss.

Let's write our own Negative Log-Likelihood function and optimize it using SciPy to prove it matches scikit-learn.

In [ ]:
def negative_log_likelihood(betas, X, y):
    # Z = X * beta
    Z = np.dot(X, betas)
    # p = sigmoid(Z)
    p = 1 / (1 + np.exp(-Z))
    # Clip to avoid log(0) errors
    p = np.clip(p, 1e-15, 1 - 1e-15)
    
    # Cross-Entropy Calculation
    nll = -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    return nll

# Initial guess for [intercept, weight_coef, age_coef]
initial_betas = np.zeros(3)
X_matrix = X_sm.values
y_vector = y_target.values

# Optimize using SciPy
res = minimize(negative_log_likelihood, initial_betas, args=(X_matrix, y_vector), method='BFGS')

print('--- MLE Optimization Results ---')
print('Optimized Betas (Custom MLE):', res.x.round(4))
print('Scikit-Learn Betas:         ', np.concatenate([logit_ml.intercept_, logit_ml.coef_[0]]).round(4))
print('\nOptimization matches production library perfectly.')

## 10. Common Mistake: Complete Separation (Hauck-Donner Effect)

If a single feature perfectly separates the 1s from the 0s, the MLE for the coefficient will attempt to reach infinity. The optimization algorithm will fail to converge or throw wildly large coefficients.

Let's simulate this edge case.

In [ ]:
# Simulate perfectly separated data
X_sep = np.linspace(-5, 5, 200).reshape(-1, 1)
y_sep = (X_sep > 0).astype(int).flatten() # Perfect cutoff at X=0

# Attempt to fit an unregularized logistic regression
sep_model = LogisticRegression(penalty=None)
sep_model.fit(X_sep, y_sep)

print('--- Complete Separation Edge Case ---')
print(f'Estimated Coefficient: {sep_model.coef_[0][0]:.2f}')
print(f'Estimated Intercept: {sep_model.intercept_[0]:.2f}')
print('\nNotice the massive coefficient! Without L2 regularization, the model attempts to push the S-curve into a perfect 90-degree step function, causing weights to explode.')

## 11. Practice Exercise: Ad Click Prediction

Scenario: You are given data on users' Time Spent on a website and whether they clicked an ad (1=Click, 0=No Click). Fit a Logit model, extract the odds ratio for Time Spent, and interpret it.

In [ ]:
# Generate Exercise Data
np.random.seed(101)
n_users = 500
time_spent = np.random.normal(15, 5, n_users) # minutes
z_ad = -4.0 + 0.3 * time_spent
clicked = np.random.binomial(1, 1 / (1 + np.exp(-z_ad)))

df_ad = pd.DataFrame({'Time_Spent': time_spent, 'Clicked': clicked})
print('Ad Data Generated. First 3 rows:')
print(df_ad.head(3))

### Solution:

In [ ]:
# 1. Fit Logit Model
X_ad = sm.add_constant(df_ad['Time_Spent'])
y_ad = df_ad['Clicked']
ad_model = sm.Logit(y_ad, X_ad).fit(disp=False)

# 2. Extract and Exponentiate Coefficient
time_coef = ad_model.params['Time_Spent']
time_odds_ratio = np.exp(time_coef)

# 3. Interpret
print('\n--- Solution ---')
print(f'Coefficient (Log-Odds): {time_coef:.4f}')
print(f'Odds Ratio: {time_odds_ratio:.4f}')
print(f'Interpretation: For every additional minute spent on the site, the odds of clicking the ad are multiplied by {time_odds_ratio:.2f}.')

## 12. Visualization Gallery: Decision Boundaries in 2D

Logistic regression forms a linear decision boundary in the feature space. Let's visualize this boundary for our beam breaking dataset.

In [ ]:
# Create a mesh grid for Weight and Age
w_min, w_max = df['Weight'].min() - 10, df['Weight'].max() + 10
a_min, a_max = df['Age'].min() - 2, df['Age'].max() + 2
ww, aa = np.meshgrid(np.linspace(w_min, w_max, 100),
                     np.linspace(a_min, a_max, 100))

# Predict probabilities for the grid
grid_preds = logit_ml.predict_proba(np.c_[ww.ravel(), aa.ravel()])[:, 1]
grid_preds = grid_preds.reshape(ww.shape)

plt.figure(figsize=(10, 7))
# Plot probability contours
contour = plt.contourf(ww, aa, grid_preds, alpha=0.8, cmap='RdYlBu_r')
plt.colorbar(contour, label='Predicted Probability P(Y=1)')

# Plot the 0.5 decision boundary line
plt.contour(ww, aa, grid_preds, levels=[0.5], colors='black', linewidths=2)

# Scatter actual data points
sns.scatterplot(x='Weight', y='Age', hue='Breaks', data=df, palette={0:'blue', 1:'red'}, edgecolor='w', s=40, alpha=0.6)
plt.title('Logit Decision Boundary (P = 0.5)')
plt.tight_layout()
plt.show()

## 13. Model Evaluation Metrics

We use metrics like Log-Loss, Accuracy, and ROC-AUC to evaluate how well our model fits the data.

In [ ]:
y_true = df['Breaks']
y_pred_prob = logit_ml.predict_proba(X_features)[:, 1]
y_pred_class = logit_ml.predict(X_features)

loss = log_loss(y_true, y_pred_prob)
acc = accuracy_score(y_true, y_pred_class)
auc = roc_auc_score(y_true, y_pred_prob)

print('--- Logit Model Performance ---')
print(f'Log-Loss (Cross-Entropy): {loss:.4f}')
print(f"Accuracy (at 0.5 thresh): {acc*100:.2f}%")
print(f'ROC-AUC Score:            {auc:.4f}')

## 14. Visualization Gallery: Comparing Logit and Probit Predictions

Let's directly compare the probability predictions of Logit and Probit for every point in our dataset.

In [ ]:
probit_preds = sm_probit.predict(X_sm)
logit_preds = sm_logit.predict(X_sm)

plt.figure(figsize=(8, 8))
plt.scatter(logit_preds, probit_preds, alpha=0.5, color='purple')
plt.plot([0, 1], [0, 1], color='black', linestyle='--') # 45 degree perfect match line
plt.title('Predicted Probabilities: Logit vs Probit')
plt.xlabel('Logit Predicted Probability')
plt.ylabel('Probit Predicted Probability')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Observation: The predictions are nearly identical, clustering tightly on the 45-degree line. The choice between Logit and Probit rarely impacts practical prediction performance.')

## 15. Summary and Key Takeaways

- The Linear Probability Model (LPM) fails because probabilities are bounded between 0 and 1, while linear functions are unbounded.
- Both Logit and Probit compress linear combinations into valid probabilities using a CDF link function.
- Logit uses the Logistic CDF. It is strictly preferred in machine learning for computational efficiency and the ability to interpret parameters as Odds Ratios.
- Probit uses the Normal CDF, motivated by normally distributed latent unobservable variables.
- Parameters are estimated using Maximum Likelihood Estimation (MLE), mathematically equivalent to optimizing Cross-Entropy Loss.
- Beware of edge cases like Complete Separation, which causes MLE coefficients to explode without regularization.

In [ ]:
print('Notebook execution complete. Logit and Probit probabilistic modeling concepts successfully demonstrated.')